In [1]:
from hydra import compose, initialize
from omegaconf import OmegaConf
import json
import os
from pathlib import Path
import glob
with initialize(config_path="dev/whole_body_benchmark/configs"):
    cfg = compose(config_name="config")

print(OmegaConf.to_yaml(cfg))

/tmp/ipykernel_20682/1456265780.py:7: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize(config_path="dev/whole_body_benchmark/configs"):


paths:
  nako_dir: /nfs/data/nii/data0/GNC/GNC_759
  nnUNet_dir: /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/
  results_dir: /nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/results/
  data_dir: /nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/data/
  oppscreen_dir: /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/data/oppscreen/



In [2]:
out_path = Path(cfg.paths.data_dir) / 'splits_966.json'
with open(out_path, 'r') as f:
    splits = json.load(f)
subjects_train, subjects_test = splits['train'], splits['test']

In [5]:
train_subset_sizes = [10, 20, 30, 50, 100, 200, 300]
dataset_name = "Dataset002_oppscreen_all_2"

# Each fold = one subset size; val = full test set for all folds
splits_final = [
    {
        "train": splits[f"train_{n}"],
        "val": splits["test"],
    }
    for n in train_subset_sizes
]

preprocessed_dir = Path(cfg.paths.nnUNet_dir) / "nnUNet_preprocessed" / dataset_name
splits_final_path = preprocessed_dir / "splits_final.json"

with open(splits_final_path, "w") as f:
    json.dump(splits_final, f, indent=2)

print(f"Wrote {len(splits_final)} folds to {splits_final_path}")
for i, (n, fold) in enumerate(zip(train_subset_sizes, splits_final)):
    print(f"  fold {i}: train_{n} ({len(fold['train'])} subjects), val ({len(fold['val'])} subjects)")

FileNotFoundError: [Errno 2] No such file or directory: '/nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/nnUNet_preprocessed/Dataset002_oppscreen_all_2/splits_final.json'

In [ ]:
img_base_path = Path(cfg.paths.nako_dir) / "links"
mask_base_path = Path(cfg.paths.nako_dir) / "data"
img_target = "30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE*_s*.nii"
mask_target = "30/opportunistic-screening/seg.nii.gz"
os.makedirs(cfg.paths.nnUNet_dir + "/nnUNet_raw", exist_ok=True)
os.makedirs(cfg.paths.nnUNet_dir + "/nnUNet_preprocessed", exist_ok=True)
os.makedirs(cfg.paths.nnUNet_dir + "/nnUNet_results", exist_ok=True)

raw_ds_dir = os.path.join(cfg.paths.nnUNet_dir, f"nnUNet_raw/{dataset_name}")
os.makedirs(raw_ds_dir, exist_ok=True)
os.makedirs(os.path.join(raw_ds_dir, "imagesTr"), exist_ok=True)
os.makedirs(os.path.join(raw_ds_dir, "labelsTr"), exist_ok=True)
os.makedirs(os.path.join(raw_ds_dir, "imagesTs"), exist_ok=True)
os.makedirs(os.path.join(raw_ds_dir, "labelsTs"), exist_ok=True)

In [12]:
import nibabel as nib
import numpy as np
import shutil

def process_subject(subject, img_base_path, mask_base_path, img_target, mask_target, images_dir, labels_dir):
    img_path = img_base_path / subject / img_target
    mask_path = mask_base_path / subject / mask_target

    # --- Images: split 4D -> 4x 3D channel files ---
    print(f"Looking for images in: {img_path}")
    files = glob.glob(str(img_path))
    print(f"Subject: {subject}, Files found: {len(files)}")

    if len(files) > 0:
        img = nib.load(files[0])
        data = img.get_fdata()  # (320, 260, 316, 4)

        if data.ndim == 4:
            for ch in range(data.shape[-1]):
                channel_data = data[..., ch].astype(np.float32)
                new_img = nib.Nifti1Image(channel_data, img.affine, img.header)
                new_img.header.set_data_shape(channel_data.shape)
                dst = os.path.join(images_dir, f"{subject}_{ch:04d}.nii.gz")
                nib.save(new_img, dst)
                print(f"  Saved channel {ch} -> {dst}")
        else:
            # skip if not 4D
            print(f"  Warning: Image is not 4D, skipping: {files[0]}")

    # --- Mask: enforce 3D header and save ---
    print(f"Looking for masks in: {mask_path}")
    mask_files = glob.glob(str(mask_path))
    print(f"Subject: {subject}, Mask files found: {len(mask_files)}")

    if len(mask_files) > 0:
        mask = nib.load(mask_files[0])
        mask_data = mask.get_fdata().astype(np.uint8)

        # Squeeze out any phantom 4th dim
        if mask_data.ndim == 4:
            mask_data = mask_data[..., 0]

        new_mask = nib.Nifti1Image(mask_data, mask.affine, mask.header)
        new_mask.header.set_data_shape(mask_data.shape)
        dst = os.path.join(labels_dir, f"{subject}.nii.gz")
        nib.save(new_mask, dst)
        print(f"  Saved mask -> {dst}")


from concurrent.futures import ProcessPoolExecutor, as_completed

def run_parallel(subjects, images_dir, labels_dir, max_workers=16):
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(
                process_subject,
                subject,
                img_base_path, mask_base_path,
                img_target, mask_target,
                images_dir, labels_dir,
            ): subject
            for subject in subjects
        }
        for future in as_completed(futures):
            subject = futures[future]
            try:
                future.result()
            except Exception as e:
                print(f"ERROR processing {subject}: {e}")

run_parallel(subjects_train,
             os.path.join(raw_ds_dir, "imagesTr"),
             os.path.join(raw_ds_dir, "labelsTr"))

run_parallel(subjects_test,
             os.path.join(raw_ds_dir, "imagesTs"),
             os.path.join(raw_ds_dir, "labelsTs"))

Looking for images in: /nfs/data/nii/data0/GNC/GNC_759/links/100537/30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE*_s*.niiLooking for images in: /nfs/data/nii/data0/GNC/GNC_759/links/100901/30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE*_s*.niiLooking for images in: /nfs/data/nii/data0/GNC/GNC_759/links/100018/30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE*_s*.niiLooking for images in: /nfs/data/nii/data0/GNC/GNC_759/links/100573/30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE*_s*.niiLooking for images in: /nfs/data/nii/data0/GNC/GNC_759/links/100360/30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE*_s*.niiLooking for images in: /nfs/data/nii/data0/GNC/GNC_759/links/100381/30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE*_s*.niiLooking for images in: /nfs/data/nii/data0/GNC/GNC_759/links/100058/30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE*_s*.niiLooking for images in: /nfs/data/nii/data0/GNC/GNC_759/links/100137/30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE*_s*.niiLooking for images in: /nfs/data/nii/data0/GNC/GNC_759/links/100452/30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSE

export nnUNet_raw="/nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/nnUNet_raw"
export nnUNet_preprocessed="/nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/nnUNet_preprocessed"
export nnUNet_results="/nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/nnUNet_results"

nnUNetv2_plan_and_preprocess -np 16 -d 2 --verify_dataset_integrity -c 3d_fullres -pl nnUNetPlannerResEncM -gpu_memory_target 40 -overwrite_plans_name nnUNetResEncUNetPlans_40G
nnUNetv2_train 2 3d_fullres -p nnUNetResEncUNetPlans_40G 3


nnUNetv2_predict \
  -i /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/nnUNet_raw/Dataset001_oppscreen_all/imagesTs \
  -o /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/nnUNet_raw/Dataset001_oppscreen_all/predsTs \
  -d 1 \
  -c 3d_fullres \
  -f all \
  -tr nnUNetTrainer \
  -p nnUNetResEncUNetPlans_40G \
  -chk checkpoint_latest.pth